In [ ]:
import json
import numpy as np
import scanpy as sc
import torch
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
dataset = 'pancreas16k'
exp_id = ''

In [ ]:
pred_path = f'./save/{dataset}/{exp_id}/cls_pred_intlabels.pt'
mapping_path = f'./save/{dataset}/{exp_id}/celltype2int.json'
true_data_path = f'./../../data/{dataset}/test.h5ad'

In [ ]:
pred_labels = torch.load(pred_path).numpy()
print(pred_labels.shape)
adata = sc.read_h5ad(true_data_path)
true_labels = adata.obs['celltype'].to_numpy()
with open(mapping_path, 'r') as f: # mapping celltype <--> int
    celltype2int = json.load(f)
int2celltype = {v: k for k, v in celltype2int.items()}
true_int_labels = np.array([celltype2int[label] for label in true_labels])
print(celltype2int)
used_labels = sorted(set(true_int_labels) | set(pred_labels))
used_target_names = [int2celltype[i] for i in used_labels] 
print(used_labels)

In [ ]:
print("Classification Report:")
print(classification_report(true_int_labels, pred_labels, labels=used_labels, target_names=used_target_names))

In [ ]:

cm = confusion_matrix(true_int_labels, pred_labels, labels=used_labels)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)
cm_normalized = np.nan_to_num(cm_normalized) 
fig, ax = plt.subplots(figsize=(12, 12), dpi=300)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=used_target_names)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, values_format=".2f", colorbar=True)
plt.xticks(rotation=45, ha='right', fontsize=10, fontweight='bold')  
plt.yticks(fontsize=10, fontweight='bold')
plt.title("Normalized Confusion Matrix", fontsize=14, fontweight='bold')
plt.xlabel("Predicted Label", fontsize=12,fontweight='bold')
plt.ylabel("True Label", fontsize=12,fontweight='bold')
plt.tight_layout()
plt.show()